# Adult Income Classification — Model Training

M.Tech (AIML/DSE) · Machine Learning · Assignment 2

This notebook trains **six classification models** on the UCI Adult Income dataset and evaluates each with six metrics (Accuracy, AUC, Precision, Recall, F1, MCC). Each fitted `Pipeline` (preprocessing + estimator) is saved to `*.joblib` for the Streamlit app.

## 1. Imports

In [1]:
import os, json
import numpy as np
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef,
    confusion_matrix, classification_report,
)

RANDOM_STATE = 42

## 2. Load & clean the data

The raw UCI file has no header. We assign column names, treat `?` as missing, drop incomplete rows, and convert the target to binary (`1` = income > 50K).

In [2]:
COLUMNS = [
    'age', 'workclass', 'fnlwgt', 'education', 'education_num',
    'marital_status', 'occupation', 'relationship', 'race', 'sex',
    'capital_gain', 'capital_loss', 'hours_per_week', 'native_country',
    'income',
]
TARGET = 'income'

df = pd.read_csv('../data/adult.data', header=None, names=COLUMNS,
                 skipinitialspace=True, na_values='?')
df = df.dropna().reset_index(drop=True)
df[TARGET] = df[TARGET].str.replace('.', '', regex=False).str.strip()
df[TARGET] = (df[TARGET] == '>50K').astype(int)

print('Shape:', df.shape, '| features:', df.shape[1]-1, '| instances:', df.shape[0])
df.head()

Shape: (30162, 15) | features: 14 | instances: 30162


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,0
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,0
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,0
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,0
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,0


## 3. Train / test split

80/20 stratified split. The 20% test split is saved as `test_data.csv` and is what the Streamlit app evaluates on.

In [3]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

test_df = X_test.copy(); test_df[TARGET] = y_test.values
test_df.to_csv('../test_data.csv', index=False)
print('train:', X_train.shape, '| test:', X_test.shape)

train: (24129, 14) | test: (6033, 14)


## 4. Preprocessing pipeline

Scale numeric features, one-hot encode categoricals. Wrapped in each model's `Pipeline` so the same transform is applied at inference.

In [4]:
categorical = X_train.select_dtypes(include=['object']).columns.tolist()
numeric = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical),
])
print('numeric:', numeric)
print('categorical:', categorical)

numeric: ['age', 'fnlwgt', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']
categorical: ['workclass', 'education', 'marital_status', 'occupation', 'relationship', 'race', 'sex', 'native_country']


## 5. Define the six models

In [5]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=RANDOM_STATE),
    'kNN': KNeighborsClassifier(n_neighbors=15),
    'Naive Bayes': GaussianNB(),
    'Random Forest (Ensemble)': RandomForestClassifier(
        n_estimators=200, max_depth=15, random_state=RANDOM_STATE, n_jobs=-1),
    'SVM': SVC(probability=True, random_state=RANDOM_STATE),
}

SLUGS = {
    'Logistic Regression': 'logistic_regression',
    'Decision Tree': 'decision_tree',
    'kNN': 'knn',
    'Naive Bayes': 'naive_bayes',
    'Random Forest (Ensemble)': 'random_forest',
    'SVM': 'svm',
}

## 6. Train, evaluate, and save each model

For every model we compute the six required metrics and persist the fitted pipeline.

In [6]:
def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_score = (model.predict_proba(X_test)[:, 1]
               if hasattr(model, 'predict_proba')
               else model.decision_function(X_test))
    return {
        'Accuracy': accuracy_score(y_test, y_pred),
        'AUC': roc_auc_score(y_test, y_score),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1': f1_score(y_test, y_pred, zero_division=0),
        'MCC': matthews_corrcoef(y_test, y_pred),
    }

rows = []
for name, est in models.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('classifier', est)])
    pipe.fit(X_train, y_train)
    m = evaluate(pipe, X_test, y_test)
    rows.append({'Model': name, **m})
    joblib.dump(pipe, f'{SLUGS[name]}.joblib')
    print(f"{name:26s} Acc={m['Accuracy']:.4f}  AUC={m['AUC']:.4f}  MCC={m['MCC']:.4f}")

Logistic Regression        Acc=0.8475  AUC=0.9022  MCC=0.5711
Decision Tree              Acc=0.8530  AUC=0.8959  MCC=0.5817


kNN                        Acc=0.8384  AUC=0.8911  MCC=0.5510
Naive Bayes                Acc=0.5826  AUC=0.8018  MCC=0.3643


Random Forest (Ensemble)   Acc=0.8556  AUC=0.9146  MCC=0.5877


SVM                        Acc=0.8498  AUC=0.8983  MCC=0.5750


## 7. Comparison table

In [7]:
metrics_df = pd.DataFrame(rows).round(4)
metrics_df.to_csv('metrics.csv', index=False)

meta = {'target': TARGET, 'feature_columns': X.columns.tolist(), 'display_names': SLUGS}
with open('meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

metrics_df.set_index('Model').style.highlight_max(axis=0, color='#c6efce')

,Accuracy,AUC,Precision,Recall,F1,MCC
Model,,,,,,
Logistic Regression,0.847500,0.902200,0.735400,0.605200,0.664000,0.571100
Decision Tree,0.853000,0.895900,0.768100,0.586600,0.665200,0.581700
kNN,0.838400,0.891100,0.700700,0.612500,0.653600,0.551000
Naive Bayes,0.582600,0.801800,0.367800,0.941400,0.529000,0.364300
Random Forest (Ensemble),0.855600,0.914600,0.785000,0.578600,0.666200,0.587700
SVM,0.849800,0.898300,0.750000,0.595200,0.663700,0.575000


## 8. Confusion matrix — best model (Random Forest)


In [8]:
best = joblib.load('random_forest.joblib')
y_pred = best.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, target_names=['<=50K', '>50K']))

[[4293  238]
 [ 633  869]]

              precision    recall  f1-score   support

       <=50K       0.87      0.95      0.91      4531
        >50K       0.79      0.58      0.67      1502

    accuracy                           0.86      6033
   macro avg       0.83      0.76      0.79      6033
weighted avg       0.85      0.86      0.85      6033



## Conclusion

**Random Forest** is the overall winner, leading on Accuracy, AUC, Precision, F1, and MCC. Naive Bayes underperforms on accuracy (its independence assumption is violated by the one-hot-encoded correlated features) but has the highest recall. All six fitted pipelines are saved as `*.joblib` and served by the Streamlit app (`../app.py`).